# COGS 138 — Neural Data Science
## Group 7 Final Project
### Are electrophysiological features from the Allen Cell Types Database reflected in resting-state EEG biomarkers for MDD?

**Group Members:** Jessica Truong, Siddhant Gulati, Tony Zheng  
**Dataset 1:** Allen Cell Types Database (mouse cortical neurons)  
**Dataset 2:** OpenNeuro ds003478 — EEG Depression Rest (122 participants)  
**Environment:** Local Mac  

---
### Project Overview
This notebook investigates whether micro-scale electrophysiological features from mouse cortical neurons in the Allen Cell Types Database correlate with macro-scale EEG biomarkers used to decode MDD in humans.

**Dataset Citation:** James F Cavanagh (2021). EEG: Depression rest. OpenNeuro. doi: 10.18112/openneuro.ds003478.v1.1.0


---
## 0. Setup — Mount Drive, Install Packages, Download Data

In [ ]:
# ============================================================
# SETUP CELL — Run this first
# ============================================================

import os
import sys

# Add Python local bin to PATH so aws works
os.environ['PATH'] += ':/Users/' + os.environ.get('USER', '') + '/Library/Python/3.12/bin'

# ============================================================
# DATA PATHS — Update if your dataset is in a different folder
# ============================================================
DATA_PATH   = os.path.expanduser('~/Desktop/Cogs138/ds003478/')
OUTPUT_PATH = os.path.expanduser('~/Desktop/Cogs138/outputs/')
ALLEN_PATH  = os.path.expanduser('~/Desktop/Cogs138/allen/')

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(ALLEN_PATH,  exist_ok=True)

# Check dataset exists
if os.path.exists(DATA_PATH):
    items = os.listdir(DATA_PATH)
    print(f'Dataset found! {len(items)} items at {DATA_PATH}')
else:
    print(f'Dataset not found at {DATA_PATH}')
    print('Run this in Terminal to download:')
    print('aws s3 sync --no-sign-request s3://openneuro.org/ds003478 ~/Desktop/Cogs138/ds003478/')

print(f'DATA_PATH   = {DATA_PATH}')
print(f'OUTPUT_PATH = {OUTPUT_PATH}')
print(f'ALLEN_PATH  = {ALLEN_PATH}')


In [ ]:
# IMPORTS CELL — Run after setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy import stats
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import mne
from mne.time_frequency import psd_array_welch
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('All libraries loaded! Running on Mac.')

---
## PART 1: OpenNeuro Dataset — Load & Explore
### 1.1 Load Participant Labels

In [ ]:
# Load participants.tsv — this has subject IDs and group labels (MDD vs healthy)
tsv_path = os.path.join(DATA_PATH, 'participants.tsv')
participants_df = pd.read_csv(tsv_path, sep='\t')

print('participants.tsv loaded!')
print(f'Total subjects: {len(participants_df)}')
print(f'Columns: {participants_df.columns.tolist()}')
print()
participants_df.head(10)

In [ ]:
# Check group distribution
# NOTE: Column name may vary — run cell above first to check
# Common column names: 'group', 'diagnosis', 'MDD', 'condition'

# Auto-detect the group column
possible_group_cols = ['group', 'diagnosis', 'MDD', 'condition', 'Group', 'Diagnosis']
group_col = None
for col in possible_group_cols:
    if col in participants_df.columns:
        group_col = col
        break

if group_col:
    print(f'Group column found: "{group_col}"')
    print(participants_df[group_col].value_counts())
else:
    print('Group column not auto-detected.')
    print('Please check participants_df.columns above and set group_col manually:')
    print('  group_col = "your_column_name"')

In [ ]:
# ============================================================
# UPDATE THIS based on what you see in participants.tsv!
# Set group_col to the column that contains information about MDD vs healthy labels.
# If a numerical score is used, set mdd_threshold for classification.
# ============================================================
group_col = 'BDI'       # Assuming BDI score is used for MDD classification
mdd_threshold = 17      # Example: BDI score >= 17 is often used for MDD

# Create binary label: 1 = MDD, 0 = healthy
participants_df['label'] = (participants_df[group_col] >= mdd_threshold).astype(int)

print('Label distribution:')
print(participants_df['label'].value_counts().rename({1: 'MDD', 0: 'Healthy'}))

### 1.2 Find EEG Files

In [ ]:
def get_eeg_files(data_path):
    """
    Walk the BIDS directory and find all EEG files.
    OpenNeuro ds003478 structure:
        sub-XXX/eeg/sub-XXX_task-Rest_run-01_eeg.set
    """
    eeg_files = []
    for root, dirs, files in os.walk(data_path):
        for f in files:
            if f.endswith('_eeg.set') or f.endswith('_eeg.edf') or f.endswith('_eeg.bdf'):
                eeg_files.append(os.path.join(root, f))
    return sorted(eeg_files)


def subject_id_from_path(filepath):
    """Extract subject ID from BIDS filepath."""
    basename = os.path.basename(filepath)
    # e.g. sub-001_task-Rest_run-01_eeg.set -> sub-001
    return basename.split('_')[0]


eeg_files = get_eeg_files(DATA_PATH)
print(f'Found {len(eeg_files)} EEG files')
print('First 5 files:')
for f in eeg_files[:5]:
    print(' ', f)

### 1.3 EEG Preprocessing Pipeline

In [ ]:
def load_and_preprocess(filepath):
    """
    Load and preprocess a single EEG file.
    Steps:
      1. Load file (EEGLAB .set format for OpenNeuro ds003478)
      2. Bandpass filter 1-40 Hz
      3. Notch filter 60 Hz (US powerline)
      4. Re-reference to average
    """
    ext = os.path.splitext(filepath)[1].lower()

    if ext == '.set':
        raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
    elif ext == '.edf':
        raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)
    elif ext == '.bdf':
        raw = mne.io.read_raw_bdf(filepath, preload=True, verbose=False)
    else:
        raise ValueError(f'Unknown format: {ext}')

    # Bandpass filter
    raw.filter(l_freq=1.0, h_freq=40.0, verbose=False)

    # Notch filter
    raw.notch_filter(freqs=60.0, verbose=False)

    # Average reference
    raw.set_eeg_reference('average', projection=False, verbose=False)

    return raw


# Test on one file
test_raw = load_and_preprocess(eeg_files[0])
print('Test file loaded successfully!')
print(test_raw.info)

### 1.4 Extract EEG Features

In [ ]:
def compute_band_power(data, sfreq, fmin, fmax):
    """Compute average power in a frequency band using Welch's method."""
    psds, freqs = psd_array_welch(
        data,
        sfreq=sfreq,
        fmin=fmin,
        fmax=fmax,
        n_fft=int(sfreq * 2),
        verbose=False
    )
    return psds.mean(axis=1)  # average across frequencies, shape: (n_channels,)


def extract_eeg_features(raw):
    """
    Extract EEG features for one subject.
    Returns dict of features.
    """
    sfreq = raw.info['sfreq']
    data = raw.get_data()  # shape: (n_channels, n_times)
    ch_names = raw.ch_names

    features = {}

    # Band powers averaged across all channels
    features['delta_power'] = compute_band_power(data, sfreq, 1, 4).mean()
    features['theta_power'] = compute_band_power(data, sfreq, 4, 8).mean()
    features['alpha_power'] = compute_band_power(data, sfreq, 8, 12).mean()
    features['beta_power']  = compute_band_power(data, sfreq, 12, 30).mean()

    # Theta/Alpha ratio — elevated in MDD
    features['theta_alpha_ratio'] = features['theta_power'] / (features['alpha_power'] + 1e-10)

    # Frontal Alpha Asymmetry (FAA) = ln(right) - ln(left)
    # Positive FAA linked to depression / withdrawal motivation
    frontal_pairs = [('F3', 'F4'), ('F7', 'F8')]
    faa_scores = []
    for left_ch, right_ch in frontal_pairs:
        if left_ch in ch_names and right_ch in ch_names:
            l_idx = ch_names.index(left_ch)
            r_idx = ch_names.index(right_ch)
            left_alpha  = compute_band_power(data[[l_idx]], sfreq, 8, 12).mean()
            right_alpha = compute_band_power(data[[r_idx]], sfreq, 8, 12).mean()
            faa = np.log(right_alpha + 1e-10) - np.log(left_alpha + 1e-10)
            faa_scores.append(faa)

    features['frontal_alpha_asymmetry'] = np.mean(faa_scores) if faa_scores else np.nan

    return features


# Test feature extraction
test_features = extract_eeg_features(test_raw)
print('Features extracted for test subject:')
for k, v in test_features.items():
    print(f'  {k}: {v:.6f}')

### 1.5 Process All Subjects

In [ ]:
all_features = []
failed = []

for i, fpath in enumerate(eeg_files):
    sub_id = subject_id_from_path(fpath)
    try:
        raw = load_and_preprocess(fpath)
        feats = extract_eeg_features(raw)
        feats['participant_id'] = sub_id
        all_features.append(feats)

        if (i + 1) % 10 == 0:
            print(f'Processed {i + 1}/{len(eeg_files)} subjects...')

    except Exception as e:
        print(f'Error with {sub_id}: {e}')
        failed.append(sub_id)

eeg_df = pd.DataFrame(all_features)

# Merge with participant labels
eeg_df = eeg_df.merge(participants_df[['participant_id', 'label']], on='participant_id', how='left')

print(f'\nSuccessfully processed: {len(eeg_df)} subjects')
print(f'Failed: {len(failed)}')
print(f'Label distribution: {eeg_df["label"].value_counts().to_dict()}')
eeg_df.head()

### 1.6 Visualize EEG Features by Group

In [ ]:
eeg_feature_cols = ['delta_power', 'theta_power', 'alpha_power',
                    'beta_power', 'theta_alpha_ratio', 'frontal_alpha_asymmetry']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('EEG Features: MDD vs Healthy Controls\n(OpenNeuro ds003478)',
             fontsize=14, fontweight='bold')

colors = {1: '#E74C3C', 0: '#2ECC71'}
labels_text = {1: 'MDD', 0: 'Healthy'}

for ax, feat in zip(axes.flatten(), eeg_feature_cols):
    for label_val, color in colors.items():
        data = eeg_df[eeg_df['label'] == label_val][feat].dropna()
        ax.hist(data, bins=20, alpha=0.6, color=color,
                label=labels_text[label_val], edgecolor='white')

    # t-test between groups
    mdd  = eeg_df[eeg_df['label'] == 1][feat].dropna()
    hc   = eeg_df[eeg_df['label'] == 0][feat].dropna()
    _, p = stats.ttest_ind(mdd, hc)

    ax.set_title(f'{feat.replace("_", " ").title()}\np = {p:.3f}', fontsize=9)
    ax.legend(fontsize=8)
    ax.set_ylabel('Count')

plt.tight_layout()
#plt.savefig(OUTPUT_PATH + 'eeg_features_comparison.png',
            #dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to Drive!')

### 1.7 Power Spectral Density Plot

In [ ]:
# Compare average PSD between MDD and healthy groups
mdd_files    = [f for f in eeg_files if subject_id_from_path(f) in
                participants_df[participants_df['label'] == 1]['participant_id'].values]
healthy_files = [f for f in eeg_files if subject_id_from_path(f) in
                participants_df[participants_df['label'] == 0]['participant_id'].values]

def avg_psd(file_list, n_max=10):
    """Compute average PSD across up to n_max subjects."""
    psds_all = []
    for fpath in file_list[:n_max]:
        try:
            raw = load_and_preprocess(fpath)
            sfreq = raw.info['sfreq']
            data = raw.get_data()
            psds, freqs = psd_array_welch(data, sfreq=sfreq,
                                           fmin=1, fmax=40,
                                           n_fft=int(sfreq*2),
                                           verbose=False)
            psds_all.append(psds.mean(axis=0))  # average across channels
        except:
            continue
    return np.array(psds_all), freqs

print('Computing PSDs... (first 10 subjects per group)')
mdd_psds, freqs    = avg_psd(mdd_files, n_max=10)
healthy_psds, _    = avg_psd(healthy_files, n_max=10)

plt.figure(figsize=(10, 5))
plt.semilogy(freqs, mdd_psds.mean(axis=0),
             color='#E74C3C', label='MDD', linewidth=2)
plt.fill_between(freqs,
                 mdd_psds.mean(0) - mdd_psds.std(0),
                 mdd_psds.mean(0) + mdd_psds.std(0),
                 alpha=0.2, color='#E74C3C')
plt.semilogy(freqs, healthy_psds.mean(axis=0),
             color='#2ECC71', label='Healthy', linewidth=2)
plt.fill_between(freqs,
                 healthy_psds.mean(0) - healthy_psds.std(0),
                 healthy_psds.mean(0) + healthy_psds.std(0),
                 alpha=0.2, color='#2ECC71')

# Mark frequency bands
for fmin, fmax, label in [(1,4,'δ'), (4,8,'θ'), (8,12,'α'), (12,30,'β')]:
    plt.axvspan(fmin, fmax, alpha=0.07, label=f'{label} band')

plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density (log)')
plt.title('Average PSD: MDD vs Healthy Controls\n(OpenNeuro ds003478)')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'psd_comparison.png', dpi=150)
plt.show()

---
## PART 2: Allen Cell Types Database
### 2.1 Load Cell Data

In [ ]:

from allensdk.core.cell_types_cache import CellTypesCache
from allensdk.ephys.ephys_extractor import EphysSweepFeatureExtractor
# Initialize Allen SDK cache
ctc = CellTypesCache(manifest_file='' + OUTPUT_PATH + 'allen_cell_types/manifest.json')

# Get all cells
cells = ctc.get_cells()
cells_df = pd.DataFrame(cells)

print(f'Total cells in Allen database: {len(cells_df)}')
cells_df.head()

In [ ]:
# Get electrophysiological features
ephys_features = ctc.get_ephys_features()
ephys_df = pd.DataFrame(ephys_features)

# Key features relevant to depression research
features_of_interest = [
    'specimen_id',
    'avg_firing_rate',
    'adaptation',
    'mean_isi',
    'cv_isi',
    'peak_v_long_square',
    'trough_v_long_square',
    'upstroke_downstroke_ratio_long_square',
    'input_resistance',
    'tau',
    'threshold_v_long_square',
    'f_i_curve_slope',
]

available = [f for f in features_of_interest if f in ephys_df.columns]
ephys_df  = ephys_df[available]

print(f'Ephys records: {len(ephys_df)}')
ephys_df.head()

In [ ]:
# Merge cell metadata with ephys features
merged_df = cells_df.merge(ephys_df,
                            left_on='id',
                            right_on='specimen_id',
                            how='inner')

# Filter for mouse cortical neurons
cortical_df = merged_df[merged_df['species'].str.contains('Mus', na=False)].copy()

print(f'Mouse cortical neurons: {len(cortical_df)}')

# Summary statistics
allen_feature_cols = ['avg_firing_rate', 'mean_isi', 'peak_v_long_square',
                      'input_resistance', 'tau', 'adaptation']
allen_feature_cols = [f for f in allen_feature_cols if f in cortical_df.columns]

cortical_df[allen_feature_cols].describe().round(3)

### 2.2 Visualize Allen Cell Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Allen Cell Types: Electrophysiological Feature Distributions\n(Mouse Cortical Neurons)',
             fontsize=13, fontweight='bold')

plot_labels = {
    'avg_firing_rate':   'Avg Firing Rate (Hz)',
    'mean_isi':          'Mean ISI (ms)',
    'peak_v_long_square':'AP Peak Voltage (mV)',
    'input_resistance':  'Input Resistance (MΩ)',
    'tau':               'Membrane Time Constant (ms)',
    'adaptation':        'Spike Frequency Adaptation',
}

for ax, feat in zip(axes.flatten(), allen_feature_cols):
    data = cortical_df[feat].dropna()
    ax.hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_xlabel(plot_labels.get(feat, feat), fontsize=9)
    ax.set_ylabel('Count')
    ax.set_title(f'{plot_labels.get(feat, feat)}\n(n={len(data)})', fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'allen_features.png', dpi=150)
plt.show()

---
## PART 3: MDD Decoding from EEG
### 3.1 Train Classifier

In [ ]:
# Prepare data — drop rows with missing labels or features
model_df = eeg_df[eeg_feature_cols + ['label']].dropna()

X = model_df[eeg_feature_cols].values
y = model_df['label'].values

print(f'Samples: {len(X)} | MDD: {y.sum()} | Healthy: {(y==0).sum()}')

# Standardize
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5-fold cross-validated Logistic Regression
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf = LogisticRegression(random_state=42, max_iter=1000)

cv_scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy')

print('\n=== Decoding Results: Logistic Regression ===')
print(f'Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print(f'Chance level: 0.500')
print(f'Per-fold: {cv_scores.round(3)}')

In [ ]:
# Also try SVM
svm = SVC(kernel='rbf', random_state=42)
svm_scores = cross_val_score(svm, X_scaled, y, cv=cv, scoring='accuracy')

print('=== Decoding Results: SVM (RBF kernel) ===')
print(f'Accuracy: {svm_scores.mean():.3f} ± {svm_scores.std():.3f}')

# Comparison bar plot
plt.figure(figsize=(6, 4))
models  = ['Logistic Regression', 'SVM (RBF)']
means   = [cv_scores.mean(), svm_scores.mean()]
stds    = [cv_scores.std(), svm_scores.std()]
colors_bar = ['#3498DB', '#9B59B6']

bars = plt.bar(models, means, yerr=stds, capsize=6,
               color=colors_bar, edgecolor='white', alpha=0.85)
plt.axhline(y=0.5, color='red', linestyle='--', label='Chance (50%)')
plt.ylabel('Cross-validated Accuracy')
plt.title('MDD Decoding Accuracy from EEG Features\n(OpenNeuro ds003478)')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'decoding_accuracy.png', dpi=150)
plt.show()

In [ ]:
# Feature importance from Logistic Regression
clf.fit(X_scaled, y)

coef_df = pd.DataFrame({
    'Feature':         eeg_feature_cols,
    'Coefficient':     clf.coef_[0],
    'Abs_Coefficient': np.abs(clf.coef_[0])
}).sort_values('Abs_Coefficient', ascending=True)

plt.figure(figsize=(8, 4))
bar_colors = ['#E74C3C' if c > 0 else '#3498DB' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'],
         color=bar_colors, edgecolor='white', alpha=0.85)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Logistic Regression Coefficient')
plt.title('EEG Feature Importance for MDD Decoding\nRed = predicts MDD | Blue = predicts Healthy')
plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'feature_importance.png', dpi=150)
plt.show()

---
## PART 4: Cross-Scale Correlation
### Allen Cell Features ↔ EEG Biomarkers

In [ ]:
# Compute population-level summary from Allen data
allen_summary = cortical_df[allen_feature_cols].describe().T[['mean', 'std']]
print('Allen Cell Types Population Summary:')
print(allen_summary.round(4))

In [ ]:
# Spearman correlation between Allen population features and EEG group differences
# Approach: for each EEG feature, compute MDD vs healthy difference score per subject
# Then correlate with Allen population statistics

# Compute group-level EEG feature means
eeg_group_means = eeg_df.groupby('label')[eeg_feature_cols].mean()
eeg_effect_sizes = eeg_group_means.loc[1] - eeg_group_means.loc[0]  # MDD - Healthy

print('EEG Feature Effect Sizes (MDD - Healthy):')
print(eeg_effect_sizes.round(4))

In [ ]:
# Spearman correlation matrix: Allen features vs EEG features
corr_results = []

for af in allen_feature_cols:
    allen_vals = cortical_df[af].dropna().values
    for ef in eeg_feature_cols:
        eeg_vals = eeg_df[ef].dropna().values
        # Correlate distributions (sample to equal length)
        n = min(len(allen_vals), len(eeg_vals))
        rho, p = spearmanr(
            np.random.choice(allen_vals, n, replace=False),
            np.random.choice(eeg_vals, n, replace=False)
        )
        corr_results.append({
            'Allen Feature': af,
            'EEG Feature': ef,
            'Spearman_rho': rho,
            'p_value': p
        })

corr_df = pd.DataFrame(corr_results)

# FDR correction
_, p_fdr, _, _ = multipletests(corr_df['p_value'], method='fdr_bh')
corr_df['p_corrected'] = p_fdr
corr_df['significant'] = corr_df['p_corrected'] < 0.05

print(f'Significant correlations after FDR correction: {corr_df["significant"].sum()} / {len(corr_df)}')
corr_df.sort_values('p_corrected').head(10)

In [ ]:
# Correlation heatmap
corr_pivot = corr_df.pivot(index='Allen Feature',
                            columns='EEG Feature',
                            values='Spearman_rho')
sig_pivot  = corr_df.pivot(index='Allen Feature',
                            columns='EEG Feature',
                            values='significant')

plt.figure(figsize=(10, 5))
ax = sns.heatmap(corr_pivot, annot=True, fmt='.2f',
                  cmap='coolwarm', center=0, vmin=-1, vmax=1,
                  linewidths=0.5, cbar_kws={'label': 'Spearman ρ'})

# Mark significant correlations
for i in range(sig_pivot.shape[0]):
    for j in range(sig_pivot.shape[1]):
        if sig_pivot.iloc[i, j]:
            ax.text(j + 0.8, i + 0.2, '*', fontsize=14,
                    color='black', fontweight='bold')

plt.title('Cross-Scale Correlation: Allen Cell Features vs Human EEG Biomarkers\n(* = significant after FDR correction, p < 0.05)',
          fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_PATH + 'cross_scale_correlation.png', dpi=150)
plt.show()

---
## PART 5: Summary

In [ ]:
print('=' * 60)
print('COGS 138 GROUP 7 — PROJECT SUMMARY')
print('=' * 60)
print(f"""
Research Question:
Are electrophysiological features from the Allen Cell Types
Database reflected in resting-state EEG biomarkers for MDD?

Datasets:
  - Allen Cell Types DB  : {len(cortical_df)} mouse cortical neurons
  - OpenNeuro ds003478   : {len(eeg_df)} subjects
                           ({int(y.sum())} MDD, {int((y==0).sum())} Healthy)

Decoding Results:
  - Logistic Regression  : {cv_scores.mean():.1%} ± {cv_scores.std():.1%}
  - SVM (RBF)            : {svm_scores.mean():.1%} ± {svm_scores.std():.1%}
  - Chance level         : 50.0%
  - Best EEG feature     : {coef_df.iloc[-1]['Feature']}

Cross-Scale Correlations:
  - Significant (FDR)    : {corr_df['significant'].sum()} / {len(corr_df)}

Limitations:
  - Cross-species (mouse → human)
  - Cross-scale (single neuron → scalp EEG)
  - Correlation ≠ causation

Outputs saved to Google Drive:
  ' + OUTPUT_PATH + '
""")
print('=' * 60)

---
## References

1. Cavanagh, J. F. (2021). EEG: Depression rest. *OpenNeuro.* doi: 10.18112/openneuro.ds003478.v1.1.0
2. Cavanagh, J. F., et al. (2019). Anger releases behavioral inhibition to cascade across response boundaries. *Psychophysiology.* PMID: 31149639
3. Gouwens, N. W., et al. (2019). Classification of electrophysiological and morphological neuron types in the mouse visual cortex. *Nature Neuroscience.*
4. Quinn, G. M. V. (2024). Resting-state EEG microstate features for MDD classification. *CUNY Academic Works.*
5. Siebenbühner, F., et al. (2026). Brain anatomy and molecular signaling predict neurophysiological dynamics. *bioRxiv.*
6. Wu, X., et al. (2021). Resting-state EEG signal for MDD detection. *Biosensors, MDPI.*
7. Yang, et al. (2023). Depression detection based on EEG signals in multi brain regions. *Journal of Integrative Neuroscience.*
